In [1]:
# Load the dataset
# Replace 'your_data.csv' with the path to your dataset
# app_df = pd.read_csv('C:/Data/Codes/Datasets/application_record.csv')
# cre_df = pd.read_csv('C:/Data/Codes/Datasets/credit_record.csv')

In [2]:
# df = pd.merge(app_df, cre_df, on='ID', how='inner')

In [3]:
# Import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# Replace 'your_data.csv' with the path to your dataset
app_df = pd.read_csv('C:/Data/Codes/Datasets/application_record.csv')
cre_df = pd.read_csv('C:/Data/Codes/Datasets/credit_record.csv')

# Load the dataset
# Replace 'your_data.csv' with the path to your dataset
df = pd.merge(app_df, cre_df, on='ID', how='inner')

# Display the first few rows of the dataset
print("Initial DataFrame Head:")
print(df.head())

# Check for missing values
print("Missing values:\n", df.isnull().sum())

# Data Cleaning: Fill or drop missing values if necessary
# Example: Uncomment the line below if you want to fill missing occupation types
# df['OCCUPATION_TYPE'].fillna('Unknown', inplace=True)

# Print the columns in the DataFrame
print("Columns in the DataFrame:")
print(df.columns.tolist())

# Check data types
print("Data types in the DataFrame:")
print(df.dtypes)

# Check unique values for all columns
for column in df.columns:
    print(f"Unique values in {column}:")
    print(df[column].unique(), "\n")

# Convert categorical variables into dummy/indicator variables
df = pd.get_dummies(df, drop_first=True)

# Display updated columns
print("Updated columns after dummy encoding:")
print(df.columns.tolist())

# Specify the target variable
target_column_name = 'STATUS_X'  # Adjust this as necessary

# Check if the target column exists
if target_column_name not in df.columns:
    print(f"{target_column_name} is not found in the DataFrame.")
else:
    # Features and target variable
    X = df.drop(columns=[target_column_name])  # Drop the target column
    y = df[target_column_name]  # The target variable represents bad credit

    # Ensure that X contains only numeric types
    X = X.select_dtypes(include=[np.number])
    print("Remaining columns data types after filtering:")
    print(X.dtypes)

    # Split the dataset into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Feature Scaling
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Model Training
    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Evaluation
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("ROC AUC Score:", roc_auc_score(y_test, y_pred))

    # Hyperparameter Tuning using GridSearchCV
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10]
    }

    grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    print("Best parameters found: ", grid_search.best_params_)
    best_model = grid_search.best_estimator_

    # Predictions with the best model
    best_y_pred = best_model.predict(X_test)

    # Evaluation of the best model
    print("Confusion Matrix of Best Model:\n", confusion_matrix(y_test, best_y_pred))
    print("Classification Report of Best Model:\n", classification_report(y_test, best_y_pred))
    print("ROC AUC Score of Best Model:", roc_auc_score(y_test, best_y_pred))


Initial DataFrame Head:
        ID CODE_GENDER FLAG_OWN_CAR FLAG_OWN_REALTY  CNT_CHILDREN  \
0  5008804           M            Y               Y             0   
1  5008804           M            Y               Y             0   
2  5008804           M            Y               Y             0   
3  5008804           M            Y               Y             0   
4  5008804           M            Y               Y             0   

   AMT_INCOME_TOTAL NAME_INCOME_TYPE NAME_EDUCATION_TYPE NAME_FAMILY_STATUS  \
0          427500.0          Working    Higher education     Civil marriage   
1          427500.0          Working    Higher education     Civil marriage   
2          427500.0          Working    Higher education     Civil marriage   
3          427500.0          Working    Higher education     Civil marriage   
4          427500.0          Working    Higher education     Civil marriage   

  NAME_HOUSING_TYPE  DAYS_BIRTH  DAYS_EMPLOYED  FLAG_MOBIL  FLAG_WORK_PHONE  \
0  Rent